# OpenCV em sistemas autônomos — 36 exemplos progressivos
Prof. Vitor Amadeu Souza · Veículos Autônomos e Robótica Móvel

Todos os exemplos têm comentário geral, instalação, procedimento e interpretação.
Use CPU: os exemplos não usam CUDA. As medições são do hardware deste runtime.
O pacote contém uma imagem KITTI e um vídeo real de direção Udacity.
Os exercícios 19 e 21 criam alterações artificiais para estudo.

No Colab, execute a configuração abaixo e envie o ZIP dos exemplos quando solicitado.
No Jupyter local, abra este caderno na pasta extraída do pacote e instale requirements.txt.


In [ ]:
from pathlib import Path
import os, sys, zipfile
try:
    import google.colab
    NO_COLAB = True
except ImportError:
    NO_COLAB = False

if NO_COLAB:
    from google.colab import files
    enviados = files.upload()  # Escolha OpenCV_Sistemas_Autonomos_36_Exemplos.zip.
    nome_zip = next((n for n in enviados if n.lower().endswith('.zip')), None)
    if nome_zip is None:
        raise ValueError('Envie o pacote ZIP dos exemplos.')
    pasta_base = Path('/content/opencv_aula')
    pasta_base.mkdir(exist_ok=True)
    with zipfile.ZipFile(nome_zip) as z:
        for membro in z.infolist():
            alvo = (pasta_base / membro.filename).resolve()
            if not alvo.is_relative_to(pasta_base.resolve()):
                raise ValueError('Caminho inválido no ZIP.')
        z.extractall(pasta_base)
    PASTA = pasta_base/'OpenCV_Sistemas_Autonomos_36_Exemplos'
else:
    PASTA = Path.cwd()
    if not (PASTA/'exemplos').is_dir():
        raise ValueError('Abra o caderno na pasta extraída do pacote.')
os.chdir(PASTA)
sys.path.insert(0, str(PASTA/'exemplos'))
print('Pasta dos exemplos:', PASTA)


In [ ]:
# Instale um único pacote OpenCV por ambiente.
# No Colab, headless evita depender de uma janela nativa.
import subprocess
arquivo_req = PASTA/('requirements-colab.txt' if NO_COLAB else 'requirements.txt')
if NO_COLAB:
    # Preparar um único OpenCV no runtime, antes de importar cv2.
    subprocess.check_call([sys.executable, '-m', 'pip', 'uninstall', '-y',
                           'opencv-python', 'opencv-contrib-python', 'opencv-contrib-python-headless'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(arquivo_req)])
from IPython.display import display, Image, Video


## 01. Conferir o ambiente de execução

Importa as bibliotecas, identifica CPU, sistema e versões e registra a configuração. Uma matriz pequena confirma que o OpenCV está funcionando.

Observe: Confirme o interpretador escolhido no VS Code. A configuração será necessária para comparar FPS.

Arquivo correspondente: `exemplos/01_ambiente.py`.


In [ ]:
# Execução independente desta célula.
__file__ = str(PASTA/'exemplos/01_ambiente.py')
sys.argv = [__file__, '--sem-gui', '--quadros', '80', '--aquecimento', '10']
"""Exemplo 01 — Conferir o ambiente de execução

COMENTÁRIO GERAL
Importa as bibliotecas, identifica CPU, sistema e versões e registra a configuração. Uma matriz pequena confirma que o OpenCV está funcionando.

O QUE INSTALAR
Python 3.10 ou superior; OpenCV, NumPy e Matplotlib.
No VS Code: python -m pip install -r requirements.txt
No Colab: executar a célula de instalação do caderno incluído no pacote.
Bibliotecas da biblioteca padrão não precisam de instalação separada.

COMO EXECUTAR
Na pasta do pacote: python exemplos/01_ambiente.py
Sem janelas: acrescentar --sem-gui. Dados próprios: --imagem ARQUIVO --video ARQUIVO.
Todas as saídas são salvas automaticamente em resultados/01_ambiente/.

O QUE OBSERVAR
Confirme o interpretador escolhido no VS Code. A configuração será necessária para comparar FPS.
"""
import numpy as np
from apoio import (iniciar, salvar_json, painel, ambiente)

a,out=iniciar(__file__)
print('Ambiente:',ambiente())
img=np.zeros((80,120,3),dtype=np.uint8)
img[:,:60]=(30,30,30);img[:,60:]=(220,220,220)
print('Matriz de teste:',img.shape,img.dtype)
salvar_json(out/'ambiente.json',ambiente())
painel([img],['Teste de leitura da matriz'],out/'teste.png',a)

# Exibir as figuras salvas na saída da célula.
for figura in sorted(out.rglob('*.png')):
    display(Image(filename=str(figura), width=850))


## 02. Carregar a imagem e imprimir os metadados

Decodifica a imagem frontal como BGR e imprime largura, altura, dtype, canais, tamanho da matriz e tamanho do arquivo. Os dois tamanhos respondem a perguntas diferentes.

Observe: Compare bytes_pixels e bytes_arquivo. A compressão do arquivo não reduz automaticamente a matriz decodificada.

Arquivo correspondente: `exemplos/02_leitura_metadados.py`.


In [ ]:
# Execução independente desta célula.
__file__ = str(PASTA/'exemplos/02_leitura_metadados.py')
sys.argv = [__file__, '--sem-gui', '--quadros', '80', '--aquecimento', '10']
"""Exemplo 02 — Carregar a imagem e imprimir os metadados

COMENTÁRIO GERAL
Decodifica a imagem frontal como BGR e imprime largura, altura, dtype, canais, tamanho da matriz e tamanho do arquivo. Os dois tamanhos respondem a perguntas diferentes.

O QUE INSTALAR
Python 3.10 ou superior; OpenCV, NumPy e Matplotlib.
No VS Code: python -m pip install -r requirements.txt
No Colab: executar a célula de instalação do caderno incluído no pacote.
Bibliotecas da biblioteca padrão não precisam de instalação separada.

COMO EXECUTAR
Na pasta do pacote: python exemplos/02_leitura_metadados.py
Sem janelas: acrescentar --sem-gui. Dados próprios: --imagem ARQUIVO --video ARQUIVO.
Todas as saídas são salvas automaticamente em resultados/02_leitura_metadados/.

O QUE OBSERVAR
Compare bytes_pixels e bytes_arquivo. A compressão do arquivo não reduz automaticamente a matriz decodificada.
"""
from apoio import (iniciar, carregar_imagem, metadados, salvar_json, painel)

a,out=iniciar(__file__)
img=carregar_imagem(a.imagem)
m=metadados(img,a.imagem)
salvar_json(out/'metadados.json',m)
painel([img],['Imagem frontal'],out/'imagem.png',a)

# Exibir as figuras salvas na saída da célula.
for figura in sorted(out.rglob('*.png')):
    display(Image(filename=str(figura), width=850))


## 03. Entender pixel, linha, coluna e recorte

Lê um pixel central e extrai uma região da metade inferior. O acesso NumPy usa [linha, coluna], enquanto o tamanho em resize usa (largura, altura).

Observe: Localize visualmente o recorte; a faixa inferior costuma concentrar o pavimento.

Arquivo correspondente: `exemplos/03_pixels_coordenadas.py`.


In [ ]:
# Execução independente desta célula.
__file__ = str(PASTA/'exemplos/03_pixels_coordenadas.py')
sys.argv = [__file__, '--sem-gui', '--quadros', '80', '--aquecimento', '10']
"""Exemplo 03 — Entender pixel, linha, coluna e recorte

COMENTÁRIO GERAL
Lê um pixel central e extrai uma região da metade inferior. O acesso NumPy usa [linha, coluna], enquanto o tamanho em resize usa (largura, altura).

O QUE INSTALAR
Python 3.10 ou superior; OpenCV, NumPy e Matplotlib.
No VS Code: python -m pip install -r requirements.txt
No Colab: executar a célula de instalação do caderno incluído no pacote.
Bibliotecas da biblioteca padrão não precisam de instalação separada.

COMO EXECUTAR
Na pasta do pacote: python exemplos/03_pixels_coordenadas.py
Sem janelas: acrescentar --sem-gui. Dados próprios: --imagem ARQUIVO --video ARQUIVO.
Todas as saídas são salvas automaticamente em resultados/03_pixels_coordenadas/.

O QUE OBSERVAR
Localize visualmente o recorte; a faixa inferior costuma concentrar o pavimento.
"""
import cv2
from apoio import (iniciar, carregar_imagem, painel)

a,out=iniciar(__file__)
img=carregar_imagem(a.imagem);h,w=img.shape[:2]
print('Centro [y,x] e valores BGR:',(h//2,w//2),img[h//2,w//2].tolist())
roi=img[h//2:h,w//4:3*w//4].copy()
marcada=img.copy()
cv2.rectangle(marcada,(w//4,h//2),(3*w//4-1,h-1),(255,255,255),2)
painel([marcada,roi],['Região escolhida','Recorte da pista'],out/'recorte.png',a)

# Exibir as figuras salvas na saída da célula.
for figura in sorted(out.rglob('*.png')):
    display(Image(filename=str(figura), width=850))


## 04. Exibir as cores corretamente

Compara a exibição incorreta de BGR diretamente no Matplotlib com a conversão BGR→RGB. O arquivo original continua BGR nas operações do OpenCV.

Observe: Veja como azul e vermelho trocam de posição quando a ordem dos canais é ignorada.

Arquivo correspondente: `exemplos/04_bgr_rgb.py`.


In [ ]:
# Execução independente desta célula.
__file__ = str(PASTA/'exemplos/04_bgr_rgb.py')
sys.argv = [__file__, '--sem-gui', '--quadros', '80', '--aquecimento', '10']
"""Exemplo 04 — Exibir as cores corretamente

COMENTÁRIO GERAL
Compara a exibição incorreta de BGR diretamente no Matplotlib com a conversão BGR→RGB. O arquivo original continua BGR nas operações do OpenCV.

O QUE INSTALAR
Python 3.10 ou superior; OpenCV, NumPy e Matplotlib.
No VS Code: python -m pip install -r requirements.txt
No Colab: executar a célula de instalação do caderno incluído no pacote.
Bibliotecas da biblioteca padrão não precisam de instalação separada.

COMO EXECUTAR
Na pasta do pacote: python exemplos/04_bgr_rgb.py
Sem janelas: acrescentar --sem-gui. Dados próprios: --imagem ARQUIVO --video ARQUIVO.
Todas as saídas são salvas automaticamente em resultados/04_bgr_rgb/.

O QUE OBSERVAR
Veja como azul e vermelho trocam de posição quando a ordem dos canais é ignorada.
"""
import cv2
import matplotlib.pyplot as plt
from apoio import (iniciar, carregar_imagem, finalizar_figura)

a,out=iniciar(__file__)
img=carregar_imagem(a.imagem)
fig,axs=plt.subplots(1,2,figsize=(11,4))
axs[0].imshow(img);axs[0].set_title('BGR tratado como RGB: cores incorretas')
axs[1].imshow(cv2.cvtColor(img,cv2.COLOR_BGR2RGB));axs[1].set_title('Conversão BGR → RGB')
for ax in axs: ax.axis('off')
finalizar_figura(fig,out/'cores.png',a)
print('BGR no OpenCV; RGB no Matplotlib.')

# Exibir as figuras salvas na saída da célula.
for figura in sorted(out.rglob('*.png')):
    display(Image(filename=str(figura), width=850))


## 05. Separar os canais B, G e R

Usa cv2.split para obter três planos de intensidade. Cada plano tem uma matriz H×W e não é, sozinho, uma fotografia colorida.

Observe: Compare a resposta da vegetação, do céu e das marcações nos três canais.

Arquivo correspondente: `exemplos/05_canais_bgr.py`.


In [ ]:
# Execução independente desta célula.
__file__ = str(PASTA/'exemplos/05_canais_bgr.py')
sys.argv = [__file__, '--sem-gui', '--quadros', '80', '--aquecimento', '10']
"""Exemplo 05 — Separar os canais B, G e R

COMENTÁRIO GERAL
Usa cv2.split para obter três planos de intensidade. Cada plano tem uma matriz H×W e não é, sozinho, uma fotografia colorida.

O QUE INSTALAR
Python 3.10 ou superior; OpenCV, NumPy e Matplotlib.
No VS Code: python -m pip install -r requirements.txt
No Colab: executar a célula de instalação do caderno incluído no pacote.
Bibliotecas da biblioteca padrão não precisam de instalação separada.

COMO EXECUTAR
Na pasta do pacote: python exemplos/05_canais_bgr.py
Sem janelas: acrescentar --sem-gui. Dados próprios: --imagem ARQUIVO --video ARQUIVO.
Todas as saídas são salvas automaticamente em resultados/05_canais_bgr/.

O QUE OBSERVAR
Compare a resposta da vegetação, do céu e das marcações nos três canais.
"""
import cv2
from apoio import (iniciar, carregar_imagem, painel)

a,out=iniciar(__file__)
img=carregar_imagem(a.imagem);b,g,r=cv2.split(img)
print('Médias B, G e R:',float(b.mean()),float(g.mean()),float(r.mean()))
painel([b,g,r],['B: intensidade de azul','G: intensidade de verde','R: intensidade de vermelho'],out/'canais.png',a)

# Exibir as figuras salvas na saída da célula.
for figura in sorted(out.rglob('*.png')):
    display(Image(filename=str(figura), width=850))


## 06. Distinguir cinza e canal V

Produz cinza por BGR2GRAY e extrai V após BGR2HSV. Cinza é uma combinação ponderada de canais; V acompanha o maior componente RGB. Não são a mesma medida.

Observe: Um objeto colorido pode ter V alto e cinza menor. Declare a definição de brilho antes de comparar regiões.

Arquivo correspondente: `exemplos/06_cinza_e_v.py`.


In [ ]:
# Execução independente desta célula.
__file__ = str(PASTA/'exemplos/06_cinza_e_v.py')
sys.argv = [__file__, '--sem-gui', '--quadros', '80', '--aquecimento', '10']
"""Exemplo 06 — Distinguir cinza e canal V

COMENTÁRIO GERAL
Produz cinza por BGR2GRAY e extrai V após BGR2HSV. Cinza é uma combinação ponderada de canais; V acompanha o maior componente RGB. Não são a mesma medida.

O QUE INSTALAR
Python 3.10 ou superior; OpenCV, NumPy e Matplotlib.
No VS Code: python -m pip install -r requirements.txt
No Colab: executar a célula de instalação do caderno incluído no pacote.
Bibliotecas da biblioteca padrão não precisam de instalação separada.

COMO EXECUTAR
Na pasta do pacote: python exemplos/06_cinza_e_v.py
Sem janelas: acrescentar --sem-gui. Dados próprios: --imagem ARQUIVO --video ARQUIVO.
Todas as saídas são salvas automaticamente em resultados/06_cinza_e_v/.

O QUE OBSERVAR
Um objeto colorido pode ter V alto e cinza menor. Declare a definição de brilho antes de comparar regiões.
"""
import cv2
from apoio import (iniciar, carregar_imagem, cinza, painel)

a,out=iniciar(__file__)
img=carregar_imagem(a.imagem);y=cinza(img)
v=cv2.cvtColor(img,cv2.COLOR_BGR2HSV)[:,:,2]
print('Média cinza:',float(y.mean()),'Média V:',float(v.mean()))
painel([img,y,v],['Original','Cinza ponderado','V = valor do HSV'],out/'cinza_v.png',a)

# Exibir as figuras salvas na saída da célula.
for figura in sorted(out.rglob('*.png')):
    display(Image(filename=str(figura), width=850))


## 07. Evitar estouro na aritmética uint8

Soma 60 à intensidade usando NumPy uint8, que pode retornar ao início da faixa, e compara com saturação explícita. A conversão para int16 permite somar antes do clip.

Observe: Verifique 240+60: o resultado esperado para aumentar brilho com saturação é 255.

Arquivo correspondente: `exemplos/07_uint8_saturacao.py`.


In [ ]:
# Execução independente desta célula.
__file__ = str(PASTA/'exemplos/07_uint8_saturacao.py')
sys.argv = [__file__, '--sem-gui', '--quadros', '80', '--aquecimento', '10']
"""Exemplo 07 — Evitar estouro na aritmética uint8

COMENTÁRIO GERAL
Soma 60 à intensidade usando NumPy uint8, que pode retornar ao início da faixa, e compara com saturação explícita. A conversão para int16 permite somar antes do clip.

O QUE INSTALAR
Python 3.10 ou superior; OpenCV, NumPy e Matplotlib.
No VS Code: python -m pip install -r requirements.txt
No Colab: executar a célula de instalação do caderno incluído no pacote.
Bibliotecas da biblioteca padrão não precisam de instalação separada.

COMO EXECUTAR
Na pasta do pacote: python exemplos/07_uint8_saturacao.py
Sem janelas: acrescentar --sem-gui. Dados próprios: --imagem ARQUIVO --video ARQUIVO.
Todas as saídas são salvas automaticamente em resultados/07_uint8_saturacao/.

O QUE OBSERVAR
Verifique 240+60: o resultado esperado para aumentar brilho com saturação é 255.
"""
import numpy as np
from apoio import (iniciar, carregar_imagem, painel)

a,out=iniciar(__file__)
img=carregar_imagem(a.imagem)
incorreta=img+np.uint8(60)
correta=np.clip(img.astype(np.int16)+60,0,255).astype(np.uint8)
print('240+60 em uint8:',(np.array([240],np.uint8)+np.uint8(60)).tolist())
print('240+60 com clip:',np.clip(np.array([240],np.int16)+60,0,255).tolist())
painel([img,incorreta,correta],['Original','Soma com retorno em uint8','Soma com saturação'],out/'uint8.png',a)

# Exibir as figuras salvas na saída da célula.
for figura in sorted(out.rglob('*.png')):
    display(Image(filename=str(figura), width=850))


## 08. Normalizar e voltar a uint8

Converte para float32, divide por 255 e reconstrói uint8 com arredondamento. Mostra a mudança no espaço de memória e no intervalo numérico.

Observe: A matriz float32 ocupa quatro vezes os bytes de uint8 para o mesmo número de elementos.

Arquivo correspondente: `exemplos/08_normalizacao.py`.


In [ ]:
# Execução independente desta célula.
__file__ = str(PASTA/'exemplos/08_normalizacao.py')
sys.argv = [__file__, '--sem-gui', '--quadros', '80', '--aquecimento', '10']
"""Exemplo 08 — Normalizar e voltar a uint8

COMENTÁRIO GERAL
Converte para float32, divide por 255 e reconstrói uint8 com arredondamento. Mostra a mudança no espaço de memória e no intervalo numérico.

O QUE INSTALAR
Python 3.10 ou superior; OpenCV, NumPy e Matplotlib.
No VS Code: python -m pip install -r requirements.txt
No Colab: executar a célula de instalação do caderno incluído no pacote.
Bibliotecas da biblioteca padrão não precisam de instalação separada.

COMO EXECUTAR
Na pasta do pacote: python exemplos/08_normalizacao.py
Sem janelas: acrescentar --sem-gui. Dados próprios: --imagem ARQUIVO --video ARQUIVO.
Todas as saídas são salvas automaticamente em resultados/08_normalizacao/.

O QUE OBSERVAR
A matriz float32 ocupa quatro vezes os bytes de uint8 para o mesmo número de elementos.
"""
import numpy as np
from apoio import (iniciar, carregar_imagem, salvar_json)

a,out=iniciar(__file__)
img=carregar_imagem(a.imagem)
f=img.astype(np.float32)/255.0
volta=np.rint(np.clip(f,0,1)*255).astype(np.uint8)
print('uint8 bytes:',img.nbytes,'float32 bytes:',f.nbytes)
print('Intervalo normalizado:',float(f.min()),float(f.max()))
print('Diferença máxima de reconstrução:',int(np.abs(img.astype(int)-volta.astype(int)).max()))
salvar_json(out/'normalizacao.json',{'uint8_bytes':img.nbytes,'float32_bytes':f.nbytes})

# Exibir as figuras salvas na saída da célula.
for figura in sorted(out.rglob('*.png')):
    display(Image(filename=str(figura), width=850))


## 09. Redimensionar e preservar proporção

Aplica o resize exigido, 640×480, e compara com um letterbox que mantém a proporção da imagem. O item B usa o resize direto; letterbox é uma alternativa discutida.

Observe: Observe a deformação de carros e compare com as margens do letterbox.

Arquivo correspondente: `exemplos/09_redimensionamento.py`.


In [ ]:
# Execução independente desta célula.
__file__ = str(PASTA/'exemplos/09_redimensionamento.py')
sys.argv = [__file__, '--sem-gui', '--quadros', '80', '--aquecimento', '10']
"""Exemplo 09 — Redimensionar e preservar proporção

COMENTÁRIO GERAL
Aplica o resize exigido, 640×480, e compara com um letterbox que mantém a proporção da imagem. O item B usa o resize direto; letterbox é uma alternativa discutida.

O QUE INSTALAR
Python 3.10 ou superior; OpenCV, NumPy e Matplotlib.
No VS Code: python -m pip install -r requirements.txt
No Colab: executar a célula de instalação do caderno incluído no pacote.
Bibliotecas da biblioteca padrão não precisam de instalação separada.

COMO EXECUTAR
Na pasta do pacote: python exemplos/09_redimensionamento.py
Sem janelas: acrescentar --sem-gui. Dados próprios: --imagem ARQUIVO --video ARQUIVO.
Todas as saídas são salvas automaticamente em resultados/09_redimensionamento/.

O QUE OBSERVAR
Observe a deformação de carros e compare com as margens do letterbox.
"""
import cv2
import numpy as np
from apoio import (iniciar, carregar_imagem, painel)

a,out=iniciar(__file__)
img=carregar_imagem(a.imagem);h,w=img.shape[:2]
direta=cv2.resize(img,(640,480),interpolation=cv2.INTER_AREA)
escala=min(640/w,480/h);nw,nh=round(w*escala),round(h*escala)
menor=cv2.resize(img,(nw,nh),interpolation=cv2.INTER_AREA)
letter=np.zeros((480,640,3),np.uint8);x,y=(640-nw)//2,(480-nh)//2
letter[y:y+nh,x:x+nw]=menor
print('Original:',(w,h),'Resize direto:',direta.shape,'Escala uniforme:',escala)
painel([direta,letter],['Resize direto 640 × 480','Proporção preservada'],out/'resize.png',a)

# Exibir as figuras salvas na saída da célula.
for figura in sorted(out.rglob('*.png')):
    display(Image(filename=str(figura), width=850))


## 10. Medir o brilho por quadrante

Converte para cinza, divide os eixos na metade e calcula a média de cada quadrante. As fatias incluem todos os pixels mesmo quando as dimensões são ímpares.

Observe: Uma diferença entre quadrantes pode vir da cena, como céu e pista, e não apenas da iluminação.

Arquivo correspondente: `exemplos/10_brilho_quadrantes.py`.


In [ ]:
# Execução independente desta célula.
__file__ = str(PASTA/'exemplos/10_brilho_quadrantes.py')
sys.argv = [__file__, '--sem-gui', '--quadros', '80', '--aquecimento', '10']
"""Exemplo 10 — Medir o brilho por quadrante

COMENTÁRIO GERAL
Converte para cinza, divide os eixos na metade e calcula a média de cada quadrante. As fatias incluem todos os pixels mesmo quando as dimensões são ímpares.

O QUE INSTALAR
Python 3.10 ou superior; OpenCV, NumPy e Matplotlib.
No VS Code: python -m pip install -r requirements.txt
No Colab: executar a célula de instalação do caderno incluído no pacote.
Bibliotecas da biblioteca padrão não precisam de instalação separada.

COMO EXECUTAR
Na pasta do pacote: python exemplos/10_brilho_quadrantes.py
Sem janelas: acrescentar --sem-gui. Dados próprios: --imagem ARQUIVO --video ARQUIVO.
Todas as saídas são salvas automaticamente em resultados/10_brilho_quadrantes/.

O QUE OBSERVAR
Uma diferença entre quadrantes pode vir da cena, como céu e pista, e não apenas da iluminação.
"""
from apoio import (iniciar, carregar_imagem, cinza, quadrantes, salvar_csv, painel)

a,out=iniciar(__file__)
img=carregar_imagem(a.imagem);y=cinza(img);q=quadrantes(y)
linhas=[{'quadrante':nome,'brilho_medio':float(reg.mean()),'pixels':reg.size} for nome,reg in q.items()]
for r in linhas: print(r)
assert sum(r['pixels'] for r in linhas)==y.size
salvar_csv(out/'quadrantes.csv',linhas)
painel(list(q.values()),[f'{r["quadrante"]}\nMédia {r["brilho_medio"]:.1f}' for r in linhas],out/'quadrantes.png',a,colunas=2)

# Exibir as figuras salvas na saída da célula.
for figura in sorted(out.rglob('*.png')):
    display(Image(filename=str(figura), width=850))


## 11. Inspecionar regiões escuras e saturadas

Calcula as frações com cinza abaixo de 30 e acima de 245 e desenha máscaras. São limiares didáticos, ajustáveis ao contexto, não uma nota universal de qualidade.

Observe: Média alta não significa boa imagem: pixels saturados podem ter perdido informação.

Arquivo correspondente: `exemplos/11_qualidade_basica.py`.


In [ ]:
# Execução independente desta célula.
__file__ = str(PASTA/'exemplos/11_qualidade_basica.py')
sys.argv = [__file__, '--sem-gui', '--quadros', '80', '--aquecimento', '10']
"""Exemplo 11 — Inspecionar regiões escuras e saturadas

COMENTÁRIO GERAL
Calcula as frações com cinza abaixo de 30 e acima de 245 e desenha máscaras. São limiares didáticos, ajustáveis ao contexto, não uma nota universal de qualidade.

O QUE INSTALAR
Python 3.10 ou superior; OpenCV, NumPy e Matplotlib.
No VS Code: python -m pip install -r requirements.txt
No Colab: executar a célula de instalação do caderno incluído no pacote.
Bibliotecas da biblioteca padrão não precisam de instalação separada.

COMO EXECUTAR
Na pasta do pacote: python exemplos/11_qualidade_basica.py
Sem janelas: acrescentar --sem-gui. Dados próprios: --imagem ARQUIVO --video ARQUIVO.
Todas as saídas são salvas automaticamente em resultados/11_qualidade_basica/.

O QUE OBSERVAR
Média alta não significa boa imagem: pixels saturados podem ter perdido informação.
"""
import numpy as np
from apoio import (iniciar, carregar_imagem, cinza, salvar_json, painel)

a,out=iniciar(__file__)
img=carregar_imagem(a.imagem);y=cinza(img)
escuro=(y<30).astype(np.uint8)*255;claro=(y>245).astype(np.uint8)*255
r={'media':float(y.mean()),'desvio':float(y.std()),'fracao_cinza_menor_30':float((y<30).mean()),'fracao_cinza_maior_245':float((y>245).mean())}
print(r);salvar_json(out/'qualidade.json',r)
painel([y,escuro,claro],['Cinza','Máscara: intensidade < 30','Máscara: intensidade > 245'],out/'qualidade.png',a)

# Exibir as figuras salvas na saída da célula.
for figura in sorted(out.rglob('*.png')):
    display(Image(filename=str(figura), width=850))


## 12. Construir um histograma de 256 níveis

Conta quantos pixels possuem cada intensidade com numpy.bincount e confirma que a soma das contagens equivale ao número de pixels.

Observe: Um histograma informa distribuição tonal, mas não a localização dos pixels na pista.

Arquivo correspondente: `exemplos/12_histograma.py`.


In [ ]:
# Execução independente desta célula.
__file__ = str(PASTA/'exemplos/12_histograma.py')
sys.argv = [__file__, '--sem-gui', '--quadros', '80', '--aquecimento', '10']
"""Exemplo 12 — Construir um histograma de 256 níveis

COMENTÁRIO GERAL
Conta quantos pixels possuem cada intensidade com numpy.bincount e confirma que a soma das contagens equivale ao número de pixels.

O QUE INSTALAR
Python 3.10 ou superior; OpenCV, NumPy e Matplotlib.
No VS Code: python -m pip install -r requirements.txt
No Colab: executar a célula de instalação do caderno incluído no pacote.
Bibliotecas da biblioteca padrão não precisam de instalação separada.

COMO EXECUTAR
Na pasta do pacote: python exemplos/12_histograma.py
Sem janelas: acrescentar --sem-gui. Dados próprios: --imagem ARQUIVO --video ARQUIVO.
Todas as saídas são salvas automaticamente em resultados/12_histograma/.

O QUE OBSERVAR
Um histograma informa distribuição tonal, mas não a localização dos pixels na pista.
"""
import numpy as np
import matplotlib.pyplot as plt
from apoio import (iniciar, carregar_imagem, cinza, salvar_csv, finalizar_figura)

a,out=iniciar(__file__)
y=cinza(carregar_imagem(a.imagem));h=np.bincount(y.ravel(),minlength=256)
print('Pixels:',y.size,'Soma do histograma:',int(h.sum()))
assert h.sum()==y.size
fig,ax=plt.subplots(figsize=(8,4));ax.plot(h,color='black')
ax.set(xlabel='Intensidade',ylabel='Quantidade de pixels',xlim=(0,255));ax.grid(alpha=.2)
finalizar_figura(fig,out/'histograma.png',a)
salvar_csv(out/'histograma.csv',[{'intensidade':i,'contagem':int(v)} for i,v in enumerate(h)])

# Exibir as figuras salvas na saída da célula.
for figura in sorted(out.rglob('*.png')):
    display(Image(filename=str(figura), width=850))


## 13. Interpretar a distribuição acumulada

Normaliza o histograma e soma as frequências progressivamente. A CDF mostra a fração de pixels com intensidade menor ou igual a cada nível.

Observe: A CDF final deve valer 1. Ela ajuda a explicar o mapeamento da equalização.

Arquivo correspondente: `exemplos/13_cdf.py`.


In [ ]:
# Execução independente desta célula.
__file__ = str(PASTA/'exemplos/13_cdf.py')
sys.argv = [__file__, '--sem-gui', '--quadros', '80', '--aquecimento', '10']
"""Exemplo 13 — Interpretar a distribuição acumulada

COMENTÁRIO GERAL
Normaliza o histograma e soma as frequências progressivamente. A CDF mostra a fração de pixels com intensidade menor ou igual a cada nível.

O QUE INSTALAR
Python 3.10 ou superior; OpenCV, NumPy e Matplotlib.
No VS Code: python -m pip install -r requirements.txt
No Colab: executar a célula de instalação do caderno incluído no pacote.
Bibliotecas da biblioteca padrão não precisam de instalação separada.

COMO EXECUTAR
Na pasta do pacote: python exemplos/13_cdf.py
Sem janelas: acrescentar --sem-gui. Dados próprios: --imagem ARQUIVO --video ARQUIVO.
Todas as saídas são salvas automaticamente em resultados/13_cdf/.

O QUE OBSERVAR
A CDF final deve valer 1. Ela ajuda a explicar o mapeamento da equalização.
"""
import numpy as np
import matplotlib.pyplot as plt
from apoio import (iniciar, carregar_imagem, cinza, finalizar_figura)

a,out=iniciar(__file__)
y=cinza(carregar_imagem(a.imagem));h=np.bincount(y.ravel(),minlength=256)/y.size
cdf=h.cumsum();print('CDF em 64,128,255:',cdf[[64,128,255]].tolist())
fig,axs=plt.subplots(1,2,figsize=(10,3.8))
axs[0].plot(h,color='black');axs[0].set(xlabel='Intensidade',ylabel='Frequência relativa')
axs[1].plot(cdf,color='black');axs[1].set(xlabel='Intensidade',ylabel='CDF',ylim=(0,1.02))
for ax in axs: ax.grid(alpha=.2)
finalizar_figura(fig,out/'cdf.png',a)

# Exibir as figuras salvas na saída da célula.
for figura in sorted(out.rglob('*.png')):
    display(Image(filename=str(figura), width=850))


## 14. Comparar equalização global e original

Usa equalizeHist sobre a imagem uint8 de um canal. O mapeamento é calculado considerando todos os pixels, o que pode ser inadequado sob sombras locais.

Observe: Inspecione separadamente regiões claras e escuras; contraste visual não comprova melhor detecção.

Arquivo correspondente: `exemplos/14_equalizacao_global.py`.


In [ ]:
# Execução independente desta célula.
__file__ = str(PASTA/'exemplos/14_equalizacao_global.py')
sys.argv = [__file__, '--sem-gui', '--quadros', '80', '--aquecimento', '10']
"""Exemplo 14 — Comparar equalização global e original

COMENTÁRIO GERAL
Usa equalizeHist sobre a imagem uint8 de um canal. O mapeamento é calculado considerando todos os pixels, o que pode ser inadequado sob sombras locais.

O QUE INSTALAR
Python 3.10 ou superior; OpenCV, NumPy e Matplotlib.
No VS Code: python -m pip install -r requirements.txt
No Colab: executar a célula de instalação do caderno incluído no pacote.
Bibliotecas da biblioteca padrão não precisam de instalação separada.

COMO EXECUTAR
Na pasta do pacote: python exemplos/14_equalizacao_global.py
Sem janelas: acrescentar --sem-gui. Dados próprios: --imagem ARQUIVO --video ARQUIVO.
Todas as saídas são salvas automaticamente em resultados/14_equalizacao_global/.

O QUE OBSERVAR
Inspecione separadamente regiões claras e escuras; contraste visual não comprova melhor detecção.
"""
import cv2
import numpy as np
import matplotlib.pyplot as plt
from apoio import (iniciar, carregar_imagem, cinza, finalizar_figura, painel)

a,out=iniciar(__file__)
y=cinza(carregar_imagem(a.imagem));eq=cv2.equalizeHist(y)
painel([y,eq],['Original em cinza','Equalização global'],out/'global.png',a)
fig,ax=plt.subplots(figsize=(8,4))
for im,label,ls in [(y,'Original','-'),(eq,'Global','--')]:
 ax.plot(np.bincount(im.ravel(),minlength=256)/im.size,color='black',label=label,linestyle=ls)
ax.set(xlabel='Intensidade',ylabel='Frequência relativa');ax.legend()
finalizar_figura(fig,out/'histogramas_global.png',a)

# Exibir as figuras salvas na saída da célula.
for figura in sorted(out.rglob('*.png')):
    display(Image(filename=str(figura), width=850))


## 15. Aplicar CLAHE à imagem em cinza

Cria CLAHE com limite de contraste 2 e grade de 8×8 regiões. Compara imagem e histogramas antes e depois usando o mesmo canal e 256 níveis.

Observe: tileGridSize=(8,8) significa oito regiões em cada eixo, e não regiões de oito pixels.

Arquivo correspondente: `exemplos/15_clahe_cinza.py`.


In [ ]:
# Execução independente desta célula.
__file__ = str(PASTA/'exemplos/15_clahe_cinza.py')
sys.argv = [__file__, '--sem-gui', '--quadros', '80', '--aquecimento', '10']
"""Exemplo 15 — Aplicar CLAHE à imagem em cinza

COMENTÁRIO GERAL
Cria CLAHE com limite de contraste 2 e grade de 8×8 regiões. Compara imagem e histogramas antes e depois usando o mesmo canal e 256 níveis.

O QUE INSTALAR
Python 3.10 ou superior; OpenCV, NumPy e Matplotlib.
No VS Code: python -m pip install -r requirements.txt
No Colab: executar a célula de instalação do caderno incluído no pacote.
Bibliotecas da biblioteca padrão não precisam de instalação separada.

COMO EXECUTAR
Na pasta do pacote: python exemplos/15_clahe_cinza.py
Sem janelas: acrescentar --sem-gui. Dados próprios: --imagem ARQUIVO --video ARQUIVO.
Todas as saídas são salvas automaticamente em resultados/15_clahe_cinza/.

O QUE OBSERVAR
tileGridSize=(8,8) significa oito regiões em cada eixo, e não regiões de oito pixels.
"""
import cv2
from apoio import (iniciar, carregar_imagem, cinza, salvar_imagem, painel, histograma_comparado)

a,out=iniciar(__file__)
y=cinza(carregar_imagem(a.imagem))
clahe=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))
eq=clahe.apply(y)
# Variações de iluminação mudam o contraste entre faixas e asfalto e entre
# obstáculos e fundo. CLAHE pode recuperar contraste local, mas amplificar
# ruído; a influência na percepção deve ser verificada, não presumida.
painel([y,eq],['Original em cinza','CLAHE: clip 2 / grade 8 × 8'],out/'clahe.png',a)
histograma_comparado(y,eq,out/'histogramas.png',a)
salvar_imagem(out/'equalizada.png',eq)

# Exibir as figuras salvas na saída da célula.
for figura in sorted(out.rglob('*.png')):
    display(Image(filename=str(figura), width=850))


## 16. Equalizar o canal L de Lab

Converte BGR para Lab, modifica apenas L por CLAHE, preserva a e b e converte de volta. Os histogramas comparam o mesmo L antes e após o algoritmo.

Observe: Não equalize separadamente B, G e R: isso pode alterar a relação entre cores.

Arquivo correspondente: `exemplos/16_clahe_colorido.py`.


In [ ]:
# Execução independente desta célula.
__file__ = str(PASTA/'exemplos/16_clahe_colorido.py')
sys.argv = [__file__, '--sem-gui', '--quadros', '80', '--aquecimento', '10']
"""Exemplo 16 — Equalizar o canal L de Lab

COMENTÁRIO GERAL
Converte BGR para Lab, modifica apenas L por CLAHE, preserva a e b e converte de volta. Os histogramas comparam o mesmo L antes e após o algoritmo.

O QUE INSTALAR
Python 3.10 ou superior; OpenCV, NumPy e Matplotlib.
No VS Code: python -m pip install -r requirements.txt
No Colab: executar a célula de instalação do caderno incluído no pacote.
Bibliotecas da biblioteca padrão não precisam de instalação separada.

COMO EXECUTAR
Na pasta do pacote: python exemplos/16_clahe_colorido.py
Sem janelas: acrescentar --sem-gui. Dados próprios: --imagem ARQUIVO --video ARQUIVO.
Todas as saídas são salvas automaticamente em resultados/16_clahe_colorido/.

O QUE OBSERVAR
Não equalize separadamente B, G e R: isso pode alterar a relação entre cores.
"""
import cv2
from apoio import (iniciar, carregar_imagem, salvar_imagem, painel, histograma_comparado)

a,out=iniciar(__file__)
img=carregar_imagem(a.imagem)
lab=cv2.cvtColor(img,cv2.COLOR_BGR2LAB);l,aa,bb=cv2.split(lab)
leq=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8)).apply(l)
eq=cv2.cvtColor(cv2.merge([leq,aa,bb]),cv2.COLOR_LAB2BGR)
painel([img,eq],['Original colorida','CLAHE apenas no canal L'],out/'clahe_lab.png',a)
histograma_comparado(l,leq,out/'histogramas_l.png',a,canal='L codificado em uint8')
salvar_imagem(out/'colorida_equalizada.png',eq)

# Exibir as figuras salvas na saída da célula.
for figura in sorted(out.rglob('*.png')):
    display(Image(filename=str(figura), width=850))


## 17. Investigar o limite de contraste do CLAHE

Mantém a grade fixa e varia apenas clipLimit entre 1, 2, 4 e 8. Isso ajuda a perceber contraste e amplificação de pequenas variações sem misturar dois parâmetros.

Observe: Mais contraste não é automaticamente melhor. Observe texturas e possíveis artefatos.

Arquivo correspondente: `exemplos/17_clahe_cliplimit.py`.


In [ ]:
# Execução independente desta célula.
__file__ = str(PASTA/'exemplos/17_clahe_cliplimit.py')
sys.argv = [__file__, '--sem-gui', '--quadros', '80', '--aquecimento', '10']
"""Exemplo 17 — Investigar o limite de contraste do CLAHE

COMENTÁRIO GERAL
Mantém a grade fixa e varia apenas clipLimit entre 1, 2, 4 e 8. Isso ajuda a perceber contraste e amplificação de pequenas variações sem misturar dois parâmetros.

O QUE INSTALAR
Python 3.10 ou superior; OpenCV, NumPy e Matplotlib.
No VS Code: python -m pip install -r requirements.txt
No Colab: executar a célula de instalação do caderno incluído no pacote.
Bibliotecas da biblioteca padrão não precisam de instalação separada.

COMO EXECUTAR
Na pasta do pacote: python exemplos/17_clahe_cliplimit.py
Sem janelas: acrescentar --sem-gui. Dados próprios: --imagem ARQUIVO --video ARQUIVO.
Todas as saídas são salvas automaticamente em resultados/17_clahe_cliplimit/.

O QUE OBSERVAR
Mais contraste não é automaticamente melhor. Observe texturas e possíveis artefatos.
"""
import cv2
from apoio import (iniciar, carregar_imagem, cinza, painel)

a,out=iniciar(__file__)
y=cinza(carregar_imagem(a.imagem));imagens=[y];titulos=['Original']
for clip in [1.,2.,4.,8.]:
 eq=cv2.createCLAHE(clipLimit=clip,tileGridSize=(8,8)).apply(y)
 imagens.append(eq);titulos.append(f'clipLimit = {clip:g}')
 print('clipLimit',clip,'média',float(eq.mean()),'desvio',float(eq.std()))
painel(imagens,titulos,out/'varrer_clip.png',a)

# Exibir as figuras salvas na saída da célula.
for figura in sorted(out.rglob('*.png')):
    display(Image(filename=str(figura), width=850))


## 18. Investigar a grade do CLAHE

Mantém clipLimit=2 e varia a quantidade de regiões: 2×2, 4×4, 8×8 e 16×16. Para uma imagem de largura W, o tamanho aproximado de cada região é W/nx.

Observe: Uma grade mais densa reage mais localmente e pode reforçar ruído ou textura.

Arquivo correspondente: `exemplos/18_clahe_grade.py`.


In [ ]:
# Execução independente desta célula.
__file__ = str(PASTA/'exemplos/18_clahe_grade.py')
sys.argv = [__file__, '--sem-gui', '--quadros', '80', '--aquecimento', '10']
"""Exemplo 18 — Investigar a grade do CLAHE

COMENTÁRIO GERAL
Mantém clipLimit=2 e varia a quantidade de regiões: 2×2, 4×4, 8×8 e 16×16. Para uma imagem de largura W, o tamanho aproximado de cada região é W/nx.

O QUE INSTALAR
Python 3.10 ou superior; OpenCV, NumPy e Matplotlib.
No VS Code: python -m pip install -r requirements.txt
No Colab: executar a célula de instalação do caderno incluído no pacote.
Bibliotecas da biblioteca padrão não precisam de instalação separada.

COMO EXECUTAR
Na pasta do pacote: python exemplos/18_clahe_grade.py
Sem janelas: acrescentar --sem-gui. Dados próprios: --imagem ARQUIVO --video ARQUIVO.
Todas as saídas são salvas automaticamente em resultados/18_clahe_grade/.

O QUE OBSERVAR
Uma grade mais densa reage mais localmente e pode reforçar ruído ou textura.
"""
import cv2
from apoio import (iniciar, carregar_imagem, cinza, painel)

a,out=iniciar(__file__)
y=cinza(carregar_imagem(a.imagem));imagens=[y];titulos=['Original'];h,w=y.shape
for n in [2,4,8,16]:
 eq=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(n,n)).apply(y)
 imagens.append(eq);titulos.append(f'Grade {n} × {n}')
 print(f'{n} × {n}: região aproximadamente {w/n:.1f} × {h/n:.1f} pixels')
painel(imagens,titulos,out/'varrer_grade.png',a)

# Exibir as figuras salvas na saída da célula.
for figura in sorted(out.rglob('*.png')):
    display(Image(filename=str(figura), width=850))


## 19. Simular uma sombra na pista

Escurece uma faixa da imagem com multiplicação em float e compara equalização global e CLAHE. A sombra é uma transformação artificial controlada de uma fotografia real.

Observe: Não interprete esta simulação como uma medição de câmera sob iluminação real.

Arquivo correspondente: `exemplos/19_iluminacao_sombra.py`.


In [ ]:
# Execução independente desta célula.
__file__ = str(PASTA/'exemplos/19_iluminacao_sombra.py')
sys.argv = [__file__, '--sem-gui', '--quadros', '80', '--aquecimento', '10']
"""Exemplo 19 — Simular uma sombra na pista

COMENTÁRIO GERAL
Escurece uma faixa da imagem com multiplicação em float e compara equalização global e CLAHE. A sombra é uma transformação artificial controlada de uma fotografia real.

O QUE INSTALAR
Python 3.10 ou superior; OpenCV, NumPy e Matplotlib.
No VS Code: python -m pip install -r requirements.txt
No Colab: executar a célula de instalação do caderno incluído no pacote.
Bibliotecas da biblioteca padrão não precisam de instalação separada.

COMO EXECUTAR
Na pasta do pacote: python exemplos/19_iluminacao_sombra.py
Sem janelas: acrescentar --sem-gui. Dados próprios: --imagem ARQUIVO --video ARQUIVO.
Todas as saídas são salvas automaticamente em resultados/19_iluminacao_sombra/.

O QUE OBSERVAR
Não interprete esta simulação como uma medição de câmera sob iluminação real.
"""
import cv2
import numpy as np
from apoio import (iniciar, carregar_imagem, cinza, painel)

a,out=iniciar(__file__)
y=cinza(carregar_imagem(a.imagem));h,w=y.shape
fator=np.ones_like(y,dtype=np.float32);fator[:,w//3:2*w//3]=.30
sombra=np.clip(y.astype(np.float32)*fator,0,255).astype(np.uint8)
glob=cv2.equalizeHist(sombra)
eq=cv2.createCLAHE(clipLimit=2,tileGridSize=(8,8)).apply(sombra)
painel([y,sombra,glob,eq],['Foto original','Sombra artificial × 0,30','Equalização global','CLAHE'],out/'sombra.png',a,colunas=2)
print('Simulação de iluminação; nenhuma informação perdida por saturação é recuperada.')

# Exibir as figuras salvas na saída da célula.
for figura in sorted(out.rglob('*.png')):
    display(Image(filename=str(figura), width=850))


## 20. Observar efeitos da compressão JPEG

Codifica e decodifica a mesma imagem com qualidades 95, 50 e 10. Mede bytes do JPEG, MSE e PSNR em relação à imagem de entrada, que já pode ter histórico de compressão.

Observe: Compare o zoom das bordas. PSNR mede diferença de pixels, não acurácia da percepção.

Arquivo correspondente: `exemplos/20_compressao_jpeg.py`.


In [ ]:
# Execução independente desta célula.
__file__ = str(PASTA/'exemplos/20_compressao_jpeg.py')
sys.argv = [__file__, '--sem-gui', '--quadros', '80', '--aquecimento', '10']
"""Exemplo 20 — Observar efeitos da compressão JPEG

COMENTÁRIO GERAL
Codifica e decodifica a mesma imagem com qualidades 95, 50 e 10. Mede bytes do JPEG, MSE e PSNR em relação à imagem de entrada, que já pode ter histórico de compressão.

O QUE INSTALAR
Python 3.10 ou superior; OpenCV, NumPy e Matplotlib.
No VS Code: python -m pip install -r requirements.txt
No Colab: executar a célula de instalação do caderno incluído no pacote.
Bibliotecas da biblioteca padrão não precisam de instalação separada.

COMO EXECUTAR
Na pasta do pacote: python exemplos/20_compressao_jpeg.py
Sem janelas: acrescentar --sem-gui. Dados próprios: --imagem ARQUIVO --video ARQUIVO.
Todas as saídas são salvas automaticamente em resultados/20_compressao_jpeg/.

O QUE OBSERVAR
Compare o zoom das bordas. PSNR mede diferença de pixels, não acurácia da percepção.
"""
import cv2
import numpy as np
from apoio import (iniciar, carregar_imagem, salvar_csv, painel)

a,out=iniciar(__file__)
img=carregar_imagem(a.imagem);imagens=[img];titulos=['Entrada'];linhas=[]
for q in [95,50,10]:
 ok,buf=cv2.imencode('.jpg',img,[cv2.IMWRITE_JPEG_QUALITY,q])
 if not ok: raise OSError('JPEG não codificado.')
 dec=cv2.imdecode(buf,cv2.IMREAD_COLOR)
 mse=float(np.mean((img.astype(np.float32)-dec.astype(np.float32))**2))
 psnr=float('inf') if mse==0 else 10*np.log10(255**2/mse)
 r={'qualidade_jpeg':q,'bytes_jpeg':len(buf),'mse':mse,'psnr_db':float(psnr)}
 linhas.append(r);print(r);imagens.append(dec);titulos.append(f'JPEG Q={q} / {len(buf):,} bytes')
 (out/f'q{q}.jpg').write_bytes(buf.tobytes())
salvar_csv(out/'jpeg.csv',linhas)
painel(imagens,titulos,out/'jpeg.png',a,colunas=2)
h,w=img.shape[:2];zoom=[im[h//3:2*h//3,w//3:2*w//3] for im in imagens]
painel(zoom,titulos,out/'jpeg_zoom.png',a,colunas=2)

# Exibir as figuras salvas na saída da célula.
for figura in sorted(out.rglob('*.png')):
    display(Image(filename=str(figura), width=850))


## 21. Adicionar ruído com semente conhecida

Cria ruído gaussiano de desvio 12 com um gerador NumPy reproduzível, soma em float, limita a faixa e volta a uint8. A imagem limpa é a referência para os exemplos seguintes.

Observe: A semente mantém o mesmo ruído para comparar filtros. O ruído é artificial.

Arquivo correspondente: `exemplos/21_ruido_gaussiano.py`.


In [ ]:
# Execução independente desta célula.
__file__ = str(PASTA/'exemplos/21_ruido_gaussiano.py')
sys.argv = [__file__, '--sem-gui', '--quadros', '80', '--aquecimento', '10']
"""Exemplo 21 — Adicionar ruído com semente conhecida

COMENTÁRIO GERAL
Cria ruído gaussiano de desvio 12 com um gerador NumPy reproduzível, soma em float, limita a faixa e volta a uint8. A imagem limpa é a referência para os exemplos seguintes.

O QUE INSTALAR
Python 3.10 ou superior; OpenCV, NumPy e Matplotlib.
No VS Code: python -m pip install -r requirements.txt
No Colab: executar a célula de instalação do caderno incluído no pacote.
Bibliotecas da biblioteca padrão não precisam de instalação separada.

COMO EXECUTAR
Na pasta do pacote: python exemplos/21_ruido_gaussiano.py
Sem janelas: acrescentar --sem-gui. Dados próprios: --imagem ARQUIVO --video ARQUIVO.
Todas as saídas são salvas automaticamente em resultados/21_ruido_gaussiano/.

O QUE OBSERVAR
A semente mantém o mesmo ruído para comparar filtros. O ruído é artificial.
"""
import numpy as np
from apoio import (iniciar, carregar_imagem, cinza, salvar_imagem, painel)

a,out=iniciar(__file__)
y=cinza(carregar_imagem(a.imagem));rng=np.random.default_rng(a.seed)
ruido=rng.normal(0,12,y.shape)
suja=np.clip(y.astype(np.float32)+ruido,0,255).astype(np.uint8)
print('Seed:',a.seed,'Desvio do ruído gerado:',float(ruido.std()))
salvar_imagem(out/'ruidosa.png',suja)
painel([y,suja],['Referência','Ruído gaussiano artificial: σ = 12'],out/'ruido.png',a)

# Exibir as figuras salvas na saída da célula.
for figura in sorted(out.rglob('*.png')):
    display(Image(filename=str(figura), width=850))


## 22. Comparar Gaussian, mediana e bilateral

Aplica três filtros ao mesmo ruído artificial e calcula MSE em relação à referência. O painel permite examinar a troca entre suavização e preservação de detalhes.

Observe: O menor MSE não garante a melhor localização de uma faixa; inspecione bordas e textura.

Arquivo correspondente: `exemplos/22_comparar_filtros.py`.


In [ ]:
# Execução independente desta célula.
__file__ = str(PASTA/'exemplos/22_comparar_filtros.py')
sys.argv = [__file__, '--sem-gui', '--quadros', '80', '--aquecimento', '10']
"""Exemplo 22 — Comparar Gaussian, mediana e bilateral

COMENTÁRIO GERAL
Aplica três filtros ao mesmo ruído artificial e calcula MSE em relação à referência. O painel permite examinar a troca entre suavização e preservação de detalhes.

O QUE INSTALAR
Python 3.10 ou superior; OpenCV, NumPy e Matplotlib.
No VS Code: python -m pip install -r requirements.txt
No Colab: executar a célula de instalação do caderno incluído no pacote.
Bibliotecas da biblioteca padrão não precisam de instalação separada.

COMO EXECUTAR
Na pasta do pacote: python exemplos/22_comparar_filtros.py
Sem janelas: acrescentar --sem-gui. Dados próprios: --imagem ARQUIVO --video ARQUIVO.
Todas as saídas são salvas automaticamente em resultados/22_comparar_filtros/.

O QUE OBSERVAR
O menor MSE não garante a melhor localização de uma faixa; inspecione bordas e textura.
"""
import cv2
import numpy as np
from apoio import (iniciar, carregar_imagem, cinza, salvar_csv, painel)

a,out=iniciar(__file__)
y=cinza(carregar_imagem(a.imagem));rng=np.random.default_rng(a.seed)
suja=np.clip(y.astype(np.float32)+rng.normal(0,12,y.shape),0,255).astype(np.uint8)
resultados={'Gaussian 5 × 5':cv2.GaussianBlur(suja,(5,5),0),
            'Mediana 5 × 5':cv2.medianBlur(suja,5),
            'Bilateral d=5':cv2.bilateralFilter(suja,5,50,50)}
linhas=[]
for nome,im in resultados.items():
 mse=float(np.mean((y.astype(np.float32)-im.astype(np.float32))**2))
 linhas.append({'filtro':nome,'mse_referencia':mse});print(nome,'MSE',mse)
salvar_csv(out/'filtros.csv',linhas)
painel([y,suja,*resultados.values()],['Referência','Ruído',*resultados.keys()],out/'filtros.png',a)

# Exibir as figuras salvas na saída da célula.
for figura in sorted(out.rglob('*.png')):
    display(Image(filename=str(figura), width=850))


## 23. Interpretar H, S e V

Converte a imagem BGR uint8 para HSV e mostra H, S e V como planos. H é reescalado apenas para visualização; o processamento conserva a codificação original.

Observe: Na conversão HSV padrão de 8 bits, H usa 0–179 e S e V usam 0–255. V não é luminância física.

Arquivo correspondente: `exemplos/23_hsv.py`.


In [ ]:
# Execução independente desta célula.
__file__ = str(PASTA/'exemplos/23_hsv.py')
sys.argv = [__file__, '--sem-gui', '--quadros', '80', '--aquecimento', '10']
"""Exemplo 23 — Interpretar H, S e V

COMENTÁRIO GERAL
Converte a imagem BGR uint8 para HSV e mostra H, S e V como planos. H é reescalado apenas para visualização; o processamento conserva a codificação original.

O QUE INSTALAR
Python 3.10 ou superior; OpenCV, NumPy e Matplotlib.
No VS Code: python -m pip install -r requirements.txt
No Colab: executar a célula de instalação do caderno incluído no pacote.
Bibliotecas da biblioteca padrão não precisam de instalação separada.

COMO EXECUTAR
Na pasta do pacote: python exemplos/23_hsv.py
Sem janelas: acrescentar --sem-gui. Dados próprios: --imagem ARQUIVO --video ARQUIVO.
Todas as saídas são salvas automaticamente em resultados/23_hsv/.

O QUE OBSERVAR
Na conversão HSV padrão de 8 bits, H usa 0–179 e S e V usam 0–255. V não é luminância física.
"""
import cv2
import numpy as np
from apoio import (iniciar, carregar_imagem, painel)

a,out=iniciar(__file__)
img=carregar_imagem(a.imagem);hsv=cv2.cvtColor(img,cv2.COLOR_BGR2HSV)
h,s,v=cv2.split(hsv)
print('Faixas observadas H/S/V:',[(int(c.min()),int(c.max())) for c in [h,s,v]])
hv=np.rint(h.astype(np.float32)*255/179).astype(np.uint8)
painel([hv,s,v],['H reescalado só para exibição','S: saturação','V: valor'],out/'hsv.png',a)

# Exibir as figuras salvas na saída da célula.
for figura in sorted(out.rglob('*.png')):
    display(Image(filename=str(figura), width=850))


## 24. Filtrar V preservando transições

Redimensiona para 640×480, converte para HSV, extrai V e aplica bilateral com d=5 e sigmas 50. Mostra a diferença absoluta amplificada entre V e o filtrado.

Observe: O bilateral combina proximidade espacial e semelhança de intensidade; parâmetros excessivos podem remover detalhes.

Arquivo correspondente: `exemplos/24_bilateral_v.py`.


In [ ]:
# Execução independente desta célula.
__file__ = str(PASTA/'exemplos/24_bilateral_v.py')
sys.argv = [__file__, '--sem-gui', '--quadros', '80', '--aquecimento', '10']
"""Exemplo 24 — Filtrar V preservando transições

COMENTÁRIO GERAL
Redimensiona para 640×480, converte para HSV, extrai V e aplica bilateral com d=5 e sigmas 50. Mostra a diferença absoluta amplificada entre V e o filtrado.

O QUE INSTALAR
Python 3.10 ou superior; OpenCV, NumPy e Matplotlib.
No VS Code: python -m pip install -r requirements.txt
No Colab: executar a célula de instalação do caderno incluído no pacote.
Bibliotecas da biblioteca padrão não precisam de instalação separada.

COMO EXECUTAR
Na pasta do pacote: python exemplos/24_bilateral_v.py
Sem janelas: acrescentar --sem-gui. Dados próprios: --imagem ARQUIVO --video ARQUIVO.
Todas as saídas são salvas automaticamente em resultados/24_bilateral_v/.

O QUE OBSERVAR
O bilateral combina proximidade espacial e semelhança de intensidade; parâmetros excessivos podem remover detalhes.
"""
import cv2
import numpy as np
from apoio import (iniciar, carregar_imagem, painel)

a,out=iniciar(__file__)
img=carregar_imagem(a.imagem)
menor=cv2.resize(img,(640,480),interpolation=cv2.INTER_AREA)
hsv=cv2.cvtColor(menor,cv2.COLOR_BGR2HSV);v=hsv[:,:,2]
filtrado=cv2.bilateralFilter(v,a.d,a.sigma_cor,a.sigma_espaco)
dif=cv2.absdiff(v,filtrado);ampliada=np.clip(dif.astype(np.int16)*8,0,255).astype(np.uint8)
print('Diferença absoluta média:',float(dif.mean()))
painel([v,filtrado,ampliada],['V original','V após bilateral','Diferença × 8'],out/'bilateral.png',a)

# Exibir as figuras salvas na saída da célula.
for figura in sorted(out.rglob('*.png')):
    display(Image(filename=str(figura), width=850))


## 25. Medir o custo do diâmetro do bilateral

Varia somente d entre 3, 5 e 9, realiza cinco aquecimentos por configuração e mede vinte aplicações ao mesmo canal V. É um microbenchmark do filtro, sem leitura de vídeo.

Observe: Compare mediana e p95. Os tempos pertencem ao computador em que o código foi executado.

Arquivo correspondente: `exemplos/25_custo_bilateral.py`.


In [ ]:
# Execução independente desta célula.
__file__ = str(PASTA/'exemplos/25_custo_bilateral.py')
sys.argv = [__file__, '--sem-gui', '--quadros', '80', '--aquecimento', '10']
"""Exemplo 25 — Medir o custo do diâmetro do bilateral

COMENTÁRIO GERAL
Varia somente d entre 3, 5 e 9, realiza cinco aquecimentos por configuração e mede vinte aplicações ao mesmo canal V. É um microbenchmark do filtro, sem leitura de vídeo.

O QUE INSTALAR
Python 3.10 ou superior; OpenCV, NumPy e Matplotlib.
No VS Code: python -m pip install -r requirements.txt
No Colab: executar a célula de instalação do caderno incluído no pacote.
Bibliotecas da biblioteca padrão não precisam de instalação separada.

COMO EXECUTAR
Na pasta do pacote: python exemplos/25_custo_bilateral.py
Sem janelas: acrescentar --sem-gui. Dados próprios: --imagem ARQUIVO --video ARQUIVO.
Todas as saídas são salvas automaticamente em resultados/25_custo_bilateral/.

O QUE OBSERVAR
Compare mediana e p95. Os tempos pertencem ao computador em que o código foi executado.
"""
import cv2
import numpy as np
import matplotlib.pyplot as plt
from apoio import (iniciar, carregar_imagem, salvar_json, salvar_csv, finalizar_figura, ambiente)

from time import perf_counter_ns
a,out=iniciar(__file__)
img=cv2.resize(carregar_imagem(a.imagem),(640,480),interpolation=cv2.INTER_AREA)
v=cv2.cvtColor(img,cv2.COLOR_BGR2HSV)[:,:,2];linhas=[]
for d in [3,5,9]:
 for _ in range(5): cv2.bilateralFilter(v,d,50,50)
 tempos=[]
 for _ in range(20):
  t0=perf_counter_ns();cv2.bilateralFilter(v,d,50,50);tempos.append((perf_counter_ns()-t0)/1e6)
 r={'d':d,'mediana_ms':float(np.median(tempos)),'p95_ms':float(np.percentile(tempos,95)),'media_ms':float(np.mean(tempos))}
 linhas.append(r);print(r)
salvar_csv(out/'custo_d.csv',linhas);salvar_json(out/'ambiente.json',ambiente())
fig,ax=plt.subplots(figsize=(7,4));ax.bar([str(r['d']) for r in linhas],[r['mediana_ms'] for r in linhas],color='0.55')
ax.set(xlabel='Diâmetro d',ylabel='Mediana do tempo (ms)');finalizar_figura(fig,out/'custo_d.png',a)

# Exibir as figuras salvas na saída da célula.
for figura in sorted(out.rglob('*.png')):
    display(Image(filename=str(figura), width=850))


## 26. Abrir o vídeo e ler o primeiro frame

Verifica a abertura de VideoCapture, lê FPS nominal, número de frames e dimensões e exibe o primeiro frame. A captura é liberada mesmo em caso de erro.

Observe: O FPS informado pelo arquivo é diferente da capacidade de processamento do algoritmo.

Arquivo correspondente: `exemplos/26_abrir_video.py`.


In [ ]:
# Execução independente desta célula.
__file__ = str(PASTA/'exemplos/26_abrir_video.py')
sys.argv = [__file__, '--sem-gui', '--quadros', '80', '--aquecimento', '10']
"""Exemplo 26 — Abrir o vídeo e ler o primeiro frame

COMENTÁRIO GERAL
Verifica a abertura de VideoCapture, lê FPS nominal, número de frames e dimensões e exibe o primeiro frame. A captura é liberada mesmo em caso de erro.

O QUE INSTALAR
Python 3.10 ou superior; OpenCV, NumPy e Matplotlib.
No VS Code: python -m pip install -r requirements.txt
No Colab: executar a célula de instalação do caderno incluído no pacote.
Bibliotecas da biblioteca padrão não precisam de instalação separada.

COMO EXECUTAR
Na pasta do pacote: python exemplos/26_abrir_video.py
Sem janelas: acrescentar --sem-gui. Dados próprios: --imagem ARQUIVO --video ARQUIVO.
Todas as saídas são salvas automaticamente em resultados/26_abrir_video/.

O QUE OBSERVAR
O FPS informado pelo arquivo é diferente da capacidade de processamento do algoritmo.
"""
import cv2
from apoio import (iniciar, abrir_video, salvar_json, painel)

a,out=iniciar(__file__)
cap=abrir_video(a.video)
try:
 m={'fps_nominal':cap.get(cv2.CAP_PROP_FPS),'frames_declarados':int(cap.get(cv2.CAP_PROP_FRAME_COUNT)),
    'largura':int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),'altura':int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))}
 print(m);salvar_json(out/'video.json',m)
 ok,frame=cap.read()
 if not ok: raise ValueError('Primeiro frame não decodificado.')
 painel([frame],['Primeiro frame do vídeo real'],out/'primeiro_frame.png',a)
finally:
 cap.release()

# Exibir as figuras salvas na saída da célula.
for figura in sorted(out.rglob('*.png')):
    display(Image(filename=str(figura), width=850))


## 27. Ler frames sequencialmente

Percorre o vídeo até o limite ou o fim, calcula a média de V por frame e desenha a evolução temporal. Este exemplo introduz o loop antes de acrescentar o filtro.

Observe: Mudanças de brilho também podem vir de uma mudança da cena. Não atribua toda variação à exposição.

Arquivo correspondente: `exemplos/27_iterar_frames.py`.


In [ ]:
# Execução independente desta célula.
__file__ = str(PASTA/'exemplos/27_iterar_frames.py')
sys.argv = [__file__, '--sem-gui', '--quadros', '80', '--aquecimento', '10']
"""Exemplo 27 — Ler frames sequencialmente

COMENTÁRIO GERAL
Percorre o vídeo até o limite ou o fim, calcula a média de V por frame e desenha a evolução temporal. Este exemplo introduz o loop antes de acrescentar o filtro.

O QUE INSTALAR
Python 3.10 ou superior; OpenCV, NumPy e Matplotlib.
No VS Code: python -m pip install -r requirements.txt
No Colab: executar a célula de instalação do caderno incluído no pacote.
Bibliotecas da biblioteca padrão não precisam de instalação separada.

COMO EXECUTAR
Na pasta do pacote: python exemplos/27_iterar_frames.py
Sem janelas: acrescentar --sem-gui. Dados próprios: --imagem ARQUIVO --video ARQUIVO.
Todas as saídas são salvas automaticamente em resultados/27_iterar_frames/.

O QUE OBSERVAR
Mudanças de brilho também podem vir de uma mudança da cena. Não atribua toda variação à exposição.
"""
import cv2
import matplotlib.pyplot as plt
from apoio import (iniciar, abrir_video, salvar_csv, finalizar_figura)

a,out=iniciar(__file__)
cap=abrir_video(a.video);linhas=[]
try:
 for i in range(a.quadros):
  ok,frame=cap.read()
  if not ok: break
  v=cv2.cvtColor(frame,cv2.COLOR_BGR2HSV)[:,:,2]
  r={'frame':i,'media_v':float(v.mean())};linhas.append(r);print(r)
finally:
 cap.release()
salvar_csv(out/'brilho_video.csv',linhas)
fig,ax=plt.subplots(figsize=(8,4));ax.plot([r['frame'] for r in linhas],[r['media_v'] for r in linhas],color='black')
ax.set(xlabel='Frame',ylabel='Média V (0–255)');ax.grid(alpha=.2)
finalizar_figura(fig,out/'brilho_video.png',a)

# Exibir as figuras salvas na saída da célula.
for figura in sorted(out.rglob('*.png')):
    display(Image(filename=str(figura), width=850))


## 28. Consolidar todas as saídas do item A

Executa leitura, metadados, brilho por quadrantes, CLAHE em cinza, histogramas sobrepostos e comparação visual. Também oferece CLAHE em Lab como extensão colorida.

Observe: Confira os JSON, PNG e o terminal. Use uma imagem frontal KITTI, nuScenes ou própria para a entrega.

Arquivo correspondente: `exemplos/28_item_a_completo.py`.


In [ ]:
# Execução independente desta célula.
__file__ = str(PASTA/'exemplos/28_item_a_completo.py')
sys.argv = [__file__, '--sem-gui', '--quadros', '80', '--aquecimento', '10']
"""Exemplo 28 — Consolidar todas as saídas do item A

COMENTÁRIO GERAL
Executa leitura, metadados, brilho por quadrantes, CLAHE em cinza, histogramas sobrepostos e comparação visual. Também oferece CLAHE em Lab como extensão colorida.

O QUE INSTALAR
Python 3.10 ou superior; OpenCV, NumPy e Matplotlib.
No VS Code: python -m pip install -r requirements.txt
No Colab: executar a célula de instalação do caderno incluído no pacote.
Bibliotecas da biblioteca padrão não precisam de instalação separada.

COMO EXECUTAR
Na pasta do pacote: python exemplos/28_item_a_completo.py
Sem janelas: acrescentar --sem-gui. Dados próprios: --imagem ARQUIVO --video ARQUIVO.
Todas as saídas são salvas automaticamente em resultados/28_item_a_completo/.

O QUE OBSERVAR
Confira os JSON, PNG e o terminal. Use uma imagem frontal KITTI, nuScenes ou própria para a entrega.
"""
from apoio import (iniciar, item_a)

a,out=iniciar(__file__)
# A função item_a reúne as operações estudadas nos exemplos 02, 10, 15 e 16.
# Em sombras, faixas e obstáculos perdem contraste; sob saturação, detalhes
# desaparecem. CLAHE pode ajudar localmente e também realçar ruído. A
# validação final depende da qualidade e do desempenho da percepção.
rel=item_a(a,out)
print('Item A concluído em:',out)
print('Definição de brilho:',rel['definicao_brilho'])

# Exibir as figuras salvas na saída da célula.
for figura in sorted(out.rglob('*.png')):
    display(Image(filename=str(figura), width=850))


## 29. Executar o pipeline completo em vídeo

Lê um vídeo real, executa resize→HSV→V→bilateral, descarta o aquecimento, imprime ms e FPS por frame e grava um feed lado a lado. Os tempos de cada etapa ficam no CSV.

Observe: Use --mostrar --sem-gui para um feed ao vivo no desktop; no Colab use o vídeo salvo. O FPS final do loop inclui gravação e impressão.

Arquivo correspondente: `exemplos/29_item_b_stream.py`.


In [ ]:
# Execução independente desta célula.
__file__ = str(PASTA/'exemplos/29_item_b_stream.py')
sys.argv = [__file__, '--sem-gui', '--quadros', '80', '--aquecimento', '10']
"""Exemplo 29 — Executar o pipeline completo em vídeo

COMENTÁRIO GERAL
Lê um vídeo real, executa resize→HSV→V→bilateral, descarta o aquecimento, imprime ms e FPS por frame e grava um feed lado a lado. Os tempos de cada etapa ficam no CSV.

O QUE INSTALAR
Python 3.10 ou superior; OpenCV, NumPy e Matplotlib.
No VS Code: python -m pip install -r requirements.txt
No Colab: executar a célula de instalação do caderno incluído no pacote.
Bibliotecas da biblioteca padrão não precisam de instalação separada.

COMO EXECUTAR
Na pasta do pacote: python exemplos/29_item_b_stream.py
Sem janelas: acrescentar --sem-gui. Dados próprios: --imagem ARQUIVO --video ARQUIVO.
Todas as saídas são salvas automaticamente em resultados/29_item_b_stream/.

O QUE OBSERVAR
Use --mostrar --sem-gui para um feed ao vivo no desktop; no Colab use o vídeo salvo. O FPS final do loop inclui gravação e impressão.
"""
from apoio import (iniciar, rodar_stream)

a,out=iniciar(__file__)
# Pipeline exigido: 640×480 → HSV → V → bilateral.
# rodar_stream trata VideoCapture/VideoWriter, liberação de recursos e medição.
# A GPU do Colab não acelera automaticamente estas chamadas cv2 na CPU.
linhas,resumo=rodar_stream(a,out,gravar=True,verbose=True)
print('Feed:',out/'feed_processado.mp4')
print('FPS processamento:',resumo['fps_processamento'],'FPS loop:',resumo['fps_loop'])

# Exibir as figuras salvas na saída da célula.
for figura in sorted(out.rglob('*.png')):
    display(Image(filename=str(figura), width=850))

# MP4V nem sempre é reproduzido pelo navegador. Converta para H.264 no Colab.
import shutil
feed = out/('item_b/feed_processado.mp4' if 'projeto_integrador' in out.name else 'feed_processado.mp4')
if feed.exists() and shutil.which('ffmpeg'):
    navegador = feed.with_name('feed_navegador.mp4')
    subprocess.run(['ffmpeg','-y','-loglevel','error','-i',str(feed),'-c:v','libx264','-pix_fmt','yuv420p','-movflags','+faststart',str(navegador)],check=True)
    display(Video(str(navegador), embed=True, width=900))
else:
    print('Feed salvo:', feed, '| Abra no VLC ou instale FFmpeg para exibir no navegador.')


## 30. Encontrar a etapa que consome mais tempo

Executa o mesmo pipeline sem gravar vídeo e sem imprimir cada frame, calcula médias por etapa e apresenta latência por frame e o prazo da meta de FPS.

Observe: A medição sem gravação é comparável apenas com outra execução sob o mesmo protocolo.

Arquivo correspondente: `exemplos/30_benchmark_etapas.py`.


In [ ]:
# Execução independente desta célula.
__file__ = str(PASTA/'exemplos/30_benchmark_etapas.py')
sys.argv = [__file__, '--sem-gui', '--quadros', '80', '--aquecimento', '10']
"""Exemplo 30 — Encontrar a etapa que consome mais tempo

COMENTÁRIO GERAL
Executa o mesmo pipeline sem gravar vídeo e sem imprimir cada frame, calcula médias por etapa e apresenta latência por frame e o prazo da meta de FPS.

O QUE INSTALAR
Python 3.10 ou superior; OpenCV, NumPy e Matplotlib.
No VS Code: python -m pip install -r requirements.txt
No Colab: executar a célula de instalação do caderno incluído no pacote.
Bibliotecas da biblioteca padrão não precisam de instalação separada.

COMO EXECUTAR
Na pasta do pacote: python exemplos/30_benchmark_etapas.py
Sem janelas: acrescentar --sem-gui. Dados próprios: --imagem ARQUIVO --video ARQUIVO.
Todas as saídas são salvas automaticamente em resultados/30_benchmark_etapas/.

O QUE OBSERVAR
A medição sem gravação é comparável apenas com outra execução sob o mesmo protocolo.
"""
import numpy as np
import matplotlib.pyplot as plt
from apoio import (iniciar, finalizar_figura, rodar_stream)

a,out=iniciar(__file__)
linhas,resumo=rodar_stream(a,out,gravar=False,verbose=False)
etapas=['resize_ms','hsv_ms','extracao_v_ms','bilateral_ms']
medias=[np.mean([float(r[k]) for r in linhas]) for k in etapas]
fig,axs=plt.subplots(1,2,figsize=(11,4))
axs[0].bar(['Resize','HSV','Extrair V','Bilateral'],medias,color='0.55');axs[0].set_ylabel('Média (ms)')
axs[1].plot([r['frame'] for r in linhas],[r['loop_ms'] for r in linhas],color='black',label='Loop')
axs[1].axhline(1000/a.fps_alvo,color='0.4',linestyle='--',label=f'Prazo: {a.fps_alvo:g} FPS')
axs[1].set(xlabel='Frame',ylabel='Tempo (ms)');axs[1].legend()
finalizar_figura(fig,out/'etapas_prazo.png',a)

# Exibir as figuras salvas na saída da célula.
for figura in sorted(out.rglob('*.png')):
    display(Image(filename=str(figura), width=850))


## 31. Comparar resolução e diâmetro

Carrega os mesmos frames em memória e mede nove configurações, combinando três resoluções e três diâmetros. Evita misturar leitura e gravação com o custo do pipeline.

Observe: 320×240 e 960×720 são extensões de estudo; a resolução exigida no item B continua 640×480.

Arquivo correspondente: `exemplos/31_comparar_configuracoes.py`.


In [ ]:
# Execução independente desta célula.
__file__ = str(PASTA/'exemplos/31_comparar_configuracoes.py')
sys.argv = [__file__, '--sem-gui', '--quadros', '80', '--aquecimento', '10']
"""Exemplo 31 — Comparar resolução e diâmetro

COMENTÁRIO GERAL
Carrega os mesmos frames em memória e mede nove configurações, combinando três resoluções e três diâmetros. Evita misturar leitura e gravação com o custo do pipeline.

O QUE INSTALAR
Python 3.10 ou superior; OpenCV, NumPy e Matplotlib.
No VS Code: python -m pip install -r requirements.txt
No Colab: executar a célula de instalação do caderno incluído no pacote.
Bibliotecas da biblioteca padrão não precisam de instalação separada.

COMO EXECUTAR
Na pasta do pacote: python exemplos/31_comparar_configuracoes.py
Sem janelas: acrescentar --sem-gui. Dados próprios: --imagem ARQUIVO --video ARQUIVO.
Todas as saídas são salvas automaticamente em resultados/31_comparar_configuracoes/.

O QUE OBSERVAR
320×240 e 960×720 são extensões de estudo; a resolução exigida no item B continua 640×480.
"""
import numpy as np
import matplotlib.pyplot as plt
from apoio import (iniciar, abrir_video, salvar_json, salvar_csv, finalizar_figura, pipeline, ambiente)

a,out=iniciar(__file__)
cap=abrir_video(a.video);frames=[]
try:
 for _ in range(min(a.quadros,45)):
  ok,frame=cap.read()
  if not ok: break
  frames.append(frame)
finally:
 cap.release()
if len(frames)<2: raise ValueError('São necessários pelo menos dois frames.')
warm=min(a.aquecimento,len(frames)-1);linhas=[]
for tamanho in [(320,240),(640,480),(960,720)]:
 for d in [3,5,9]:
  tempos=[]
  for i,frame in enumerate(frames):
   _,_,_,t=pipeline(frame,tamanho,d,50,50)
   if i>=warm: tempos.append(t['proc_ms'])
  r={'largura':tamanho[0],'altura':tamanho[1],'d':d,'n':len(tempos),
     'media_ms':float(np.mean(tempos)),'p95_ms':float(np.percentile(tempos,95)),
     'fps_processamento':float(1000*len(tempos)/sum(tempos))}
  linhas.append(r);print(r)
salvar_csv(out/'configuracoes.csv',linhas);salvar_json(out/'ambiente.json',ambiente())
fig,ax=plt.subplots(figsize=(9,4))
for d,ls in [(3,'-'),(5,'--'),(9,':')]:
 rs=[r for r in linhas if r['d']==d]
 ax.plot([r['largura']*r['altura']/1e6 for r in rs],[r['media_ms'] for r in rs],label=f'd={d}',linestyle=ls,color='black',marker='o')
ax.set(xlabel='Milhões de pixels',ylabel='Média do pipeline (ms)');ax.legend();ax.grid(alpha=.2)
finalizar_figura(fig,out/'configuracoes.png',a)

# Exibir as figuras salvas na saída da célula.
for figura in sorted(out.rglob('*.png')):
    display(Image(filename=str(figura), width=850))


## 32. Relacionar contraste e bordas da pista

Aplica Canny antes e após CLAHE e uma máscara trapezoidal de região de interesse. As bordas são uma etapa ilustrativa para localização de faixas, sem medir acurácia.

Observe: Um aumento na quantidade de bordas pode representar faixas ou ruído. Não equivale a uma melhora demonstrada na detecção.

Arquivo correspondente: `exemplos/32_bordas_faixas.py`.


In [ ]:
# Execução independente desta célula.
__file__ = str(PASTA/'exemplos/32_bordas_faixas.py')
sys.argv = [__file__, '--sem-gui', '--quadros', '80', '--aquecimento', '10']
"""Exemplo 32 — Relacionar contraste e bordas da pista

COMENTÁRIO GERAL
Aplica Canny antes e após CLAHE e uma máscara trapezoidal de região de interesse. As bordas são uma etapa ilustrativa para localização de faixas, sem medir acurácia.

O QUE INSTALAR
Python 3.10 ou superior; OpenCV, NumPy e Matplotlib.
No VS Code: python -m pip install -r requirements.txt
No Colab: executar a célula de instalação do caderno incluído no pacote.
Bibliotecas da biblioteca padrão não precisam de instalação separada.

COMO EXECUTAR
Na pasta do pacote: python exemplos/32_bordas_faixas.py
Sem janelas: acrescentar --sem-gui. Dados próprios: --imagem ARQUIVO --video ARQUIVO.
Todas as saídas são salvas automaticamente em resultados/32_bordas_faixas/.

O QUE OBSERVAR
Um aumento na quantidade de bordas pode representar faixas ou ruído. Não equivale a uma melhora demonstrada na detecção.
"""
import cv2
import numpy as np
from apoio import (iniciar, carregar_imagem, cinza, painel, RAIZ)

a,out=iniciar(__file__)
# Para este exemplo de faixas, a entrada padrão é um frame do vídeo de pista.
# Uma imagem fornecida explicitamente com --imagem tem prioridade.
caminho=RAIZ/'dados/pista_udacity.png' if a.imagem==RAIZ/'dados/frontal_kitti.png' else a.imagem
y=cinza(cv2.resize(carregar_imagem(caminho),(640,480)))
eq=cv2.createCLAHE(clipLimit=2,tileGridSize=(8,8)).apply(y)
mask=np.zeros_like(y);pontos=np.array([[(30,479),(260,270),(380,270),(610,479)]],np.int32)
cv2.fillPoly(mask,pontos,255)
e0=cv2.bitwise_and(cv2.Canny(y,70,140),mask)
e1=cv2.bitwise_and(cv2.Canny(eq,70,140),mask)
print('Pixels de borda na ROI antes/depois:',int((e0>0).sum()),int((e1>0).sum()))
painel([y,eq,e0,e1],['Cinza','CLAHE','Bordas na ROI: original','Bordas na ROI: CLAHE'],out/'bordas.png',a,colunas=2)

# Exibir as figuras salvas na saída da célula.
for figura in sorted(out.rglob('*.png')):
    display(Image(filename=str(figura), width=850))


## 33. Examinar contornos sem atribuir classes

Filtra V, extrai bordas e desenha retângulos de contornos grandes na metade inferior. O exemplo mostra como ruído e sombras alteram candidatos geométricos.

Observe: Um retângulo não comprova um obstáculo. Um detector real precisa de modelo ou critérios e validação com anotações.

Arquivo correspondente: `exemplos/33_contornos_obstaculos.py`.


In [ ]:
# Execução independente desta célula.
__file__ = str(PASTA/'exemplos/33_contornos_obstaculos.py')
sys.argv = [__file__, '--sem-gui', '--quadros', '80', '--aquecimento', '10']
"""Exemplo 33 — Examinar contornos sem atribuir classes

COMENTÁRIO GERAL
Filtra V, extrai bordas e desenha retângulos de contornos grandes na metade inferior. O exemplo mostra como ruído e sombras alteram candidatos geométricos.

O QUE INSTALAR
Python 3.10 ou superior; OpenCV, NumPy e Matplotlib.
No VS Code: python -m pip install -r requirements.txt
No Colab: executar a célula de instalação do caderno incluído no pacote.
Bibliotecas da biblioteca padrão não precisam de instalação separada.

COMO EXECUTAR
Na pasta do pacote: python exemplos/33_contornos_obstaculos.py
Sem janelas: acrescentar --sem-gui. Dados próprios: --imagem ARQUIVO --video ARQUIVO.
Todas as saídas são salvas automaticamente em resultados/33_contornos_obstaculos/.

O QUE OBSERVAR
Um retângulo não comprova um obstáculo. Um detector real precisa de modelo ou critérios e validação com anotações.
"""
import cv2
from apoio import (iniciar, carregar_imagem, painel, pipeline)

a,out=iniciar(__file__)
img=carregar_imagem(a.imagem);menor,v,f,_=pipeline(img)
bordas=cv2.Canny(f,80,160);bordas[:240]=0
contornos,_=cv2.findContours(bordas,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
view=menor.copy();n=0
for c in contornos:
 x,y,w,h=cv2.boundingRect(c)
 if w*h>=600:
  cv2.rectangle(view,(x,y),(x+w,y+h),(255,255,255),1);n+=1
print('Regiões geométricas candidatas:',n,'(não são obstáculos confirmados)')
painel([menor,bordas,view],['Frame','Bordas na metade inferior','Retângulos de contornos'],out/'contornos.png',a)

# Exibir as figuras salvas na saída da célula.
for figura in sorted(out.rglob('*.png')):
    display(Image(filename=str(figura), width=850))


## 34. Gerar uma discussão baseada no CSV

Executa o fluxo, lê novamente o CSV e calcula FPS como N/tempo total, percentis e percentual acima do prazo. Gera um texto com os dados reais desta execução.

Observe: O relatório identifica o hardware e exige medição no Raspberry Pi para concluir a viabilidade nessa placa.

Arquivo correspondente: `exemplos/34_relatorio_fps.py`.


In [ ]:
# Execução independente desta célula.
__file__ = str(PASTA/'exemplos/34_relatorio_fps.py')
sys.argv = [__file__, '--sem-gui', '--quadros', '80', '--aquecimento', '10']
"""Exemplo 34 — Gerar uma discussão baseada no CSV

COMENTÁRIO GERAL
Executa o fluxo, lê novamente o CSV e calcula FPS como N/tempo total, percentis e percentual acima do prazo. Gera um texto com os dados reais desta execução.

O QUE INSTALAR
Python 3.10 ou superior; OpenCV, NumPy e Matplotlib.
No VS Code: python -m pip install -r requirements.txt
No Colab: executar a célula de instalação do caderno incluído no pacote.
Bibliotecas da biblioteca padrão não precisam de instalação separada.

COMO EXECUTAR
Na pasta do pacote: python exemplos/34_relatorio_fps.py
Sem janelas: acrescentar --sem-gui. Dados próprios: --imagem ARQUIVO --video ARQUIVO.
Todas as saídas são salvas automaticamente em resultados/34_relatorio_fps/.

O QUE OBSERVAR
O relatório identifica o hardware e exige medição no Raspberry Pi para concluir a viabilidade nessa placa.
"""
import matplotlib.pyplot as plt
from apoio import (iniciar, ler_csv, finalizar_figura, ambiente, resumo_tempos, rodar_stream)

a,out=iniciar(__file__)
rodar_stream(a,out,gravar=False,verbose=False)
linhas=ler_csv(out/'tempos_frames.csv');r=resumo_tempos(linhas,a.fps_alvo);hw=ambiente()
placa='raspberry pi 4' in hw['modelo_placa'].lower()
texto=f"""Relatório de desempenho — execução local

Hardware: {hw['cpu']}; placa: {hw['modelo_placa']}; arquitetura: {hw['arquitetura']}.
OpenCV: {hw['opencv']}; threads: {hw['threads_opencv']}; GPU: não utilizada.
Pipeline: resize 640×480, BGR→HSV, V e bilateral d={a.d}.
Frames medidos: {r['frames_medidos']}; aquecimento solicitado: {a.aquecimento}.
Média do processamento: {r['proc_media_ms']:.3f} ms; FPS do processamento: {r['fps_processamento']:.2f}.
FPS do loop: {r['fps_loop']:.2f}; p95 do loop: {r['loop_p95_ms']:.3f} ms.
Meta: {a.fps_alvo:g} FPS; prazo: {r['prazo_ms']:.3f} ms.
Frames acima do prazo: {r['frames_acima_prazo']} ({r['percentual_acima_prazo']:.2f}%).

O resultado descreve a capacidade offline deste hardware sob este protocolo.
Identificação como Raspberry Pi 4: {'sim' if placa else 'não confirmada'}.
Para avaliar essa placa, repetir a medição nela, registrar clock, temperatura,
resfriamento, SO, backend de captura e versões, e usar um teste prolongado.
Mesmo com FPS médio suficiente, verificar p95/p99 e o custo das etapas
seguintes de percepção. O teste não mede detector de faixas ou obstáculos.
"""
(out/'relatorio.txt').write_text(texto,encoding='utf-8');print(texto)
fig,ax=plt.subplots(figsize=(8,4));ms=[float(x['loop_ms']) for x in linhas]
ax.hist(ms,bins=20,color='0.65',edgecolor='white');ax.axvline(r['prazo_ms'],color='black',linestyle='--',label='Prazo')
ax.set(xlabel='Tempo do loop (ms)',ylabel='Frames');ax.legend()
finalizar_figura(fig,out/'latencia_histograma.png',a)

# Exibir as figuras salvas na saída da célula.
for figura in sorted(out.rglob('*.png')):
    display(Image(filename=str(figura), width=850))


## 35. Registrar condições da plataforma embarcada

Salva modelo da placa, CPU, versões e leituras opcionais de temperatura, clock e throttling via vcgencmd. Fora do Raspberry Pi, registra que a informação não está disponível.

Observe: Ausência de leitura não significa temperatura baixa ou ausência de throttling. Execute na placa para coletar os dados.

Arquivo correspondente: `exemplos/35_raspberry_pi.py`.


In [ ]:
# Execução independente desta célula.
__file__ = str(PASTA/'exemplos/35_raspberry_pi.py')
sys.argv = [__file__, '--sem-gui', '--quadros', '80', '--aquecimento', '10']
"""Exemplo 35 — Registrar condições da plataforma embarcada

COMENTÁRIO GERAL
Salva modelo da placa, CPU, versões e leituras opcionais de temperatura, clock e throttling via vcgencmd. Fora do Raspberry Pi, registra que a informação não está disponível.

O QUE INSTALAR
Python 3.10 ou superior; OpenCV, NumPy e Matplotlib.
No VS Code: python -m pip install -r requirements.txt
No Colab: executar a célula de instalação do caderno incluído no pacote.
Bibliotecas da biblioteca padrão não precisam de instalação separada.

COMO EXECUTAR
Na pasta do pacote: python exemplos/35_raspberry_pi.py
Sem janelas: acrescentar --sem-gui. Dados próprios: --imagem ARQUIVO --video ARQUIVO.
Todas as saídas são salvas automaticamente em resultados/35_raspberry_pi/.

O QUE OBSERVAR
Ausência de leitura não significa temperatura baixa ou ausência de throttling. Execute na placa para coletar os dados.
"""
from apoio import (iniciar, salvar_json, ambiente)

import shutil
import subprocess
a,out=iniciar(__file__)
r=ambiente();comando=shutil.which('vcgencmd')
for nome,args in [('temperatura',['measure_temp']),('clock_arm',['measure_clock','arm']),('throttling',['get_throttled'])]:
 if comando:
  try:
   p=subprocess.run([comando,*args],capture_output=True,text=True,timeout=5,check=False)
   r[nome]=p.stdout.strip() if p.returncode==0 else 'Falha na leitura: '+p.stderr.strip()
  except (OSError,subprocess.TimeoutExpired) as e: r[nome]='Indisponível: '+str(e)
 else: r[nome]='vcgencmd não disponível neste sistema'
salvar_json(out/'condicoes_hardware.json',r)
print(r)
print('Roteiro: medir por alguns minutos; não afirmar viabilidade do Pi com FPS de PC/Colab.')

# Exibir as figuras salvas na saída da célula.
for figura in sorted(out.rglob('*.png')):
    display(Image(filename=str(figura), width=850))


## 36. Executar os itens A e B e organizar a entrega

Reúne a inspeção da imagem e o processamento do vídeo em pastas separadas, produz métricas e um roteiro preenchido com os resultados medidos. É a referência final de execução.

Observe: A conclusão sobre Raspberry Pi exige dados obtidos nessa placa. Anexe mídia, plots, CSV e uma discussão técnica.

Arquivo correspondente: `exemplos/36_projeto_integrador.py`.


In [ ]:
# Execução independente desta célula.
__file__ = str(PASTA/'exemplos/36_projeto_integrador.py')
sys.argv = [__file__, '--sem-gui', '--quadros', '80', '--aquecimento', '10']
"""Exemplo 36 — Executar os itens A e B e organizar a entrega

COMENTÁRIO GERAL
Reúne a inspeção da imagem e o processamento do vídeo em pastas separadas, produz métricas e um roteiro preenchido com os resultados medidos. É a referência final de execução.

O QUE INSTALAR
Python 3.10 ou superior; OpenCV, NumPy e Matplotlib.
No VS Code: python -m pip install -r requirements.txt
No Colab: executar a célula de instalação do caderno incluído no pacote.
Bibliotecas da biblioteca padrão não precisam de instalação separada.

COMO EXECUTAR
Na pasta do pacote: python exemplos/36_projeto_integrador.py
Sem janelas: acrescentar --sem-gui. Dados próprios: --imagem ARQUIVO --video ARQUIVO.
Todas as saídas são salvas automaticamente em resultados/36_projeto_integrador/.

O QUE OBSERVAR
A conclusão sobre Raspberry Pi exige dados obtidos nessa placa. Anexe mídia, plots, CSV e uma discussão técnica.
"""
from apoio import (iniciar, rodar_stream, item_a)

a,out=iniciar(__file__)
pa=out/'item_a';pb=out/'item_b';pa.mkdir(exist_ok=True);pb.mkdir(exist_ok=True)
ra=item_a(a,pa)
linhas,rb=rodar_stream(a,pb,gravar=True,verbose=True)
# Comentário técnico: sombra reduz contraste local; brilho excessivo satura
# detalhes; JPEG pode alterar bordas. Ajustar CLAHE e bilateral exige observar
# a imagem e validar a percepção. Nenhum desses filtros classifica obstáculos.
hw=rb['ambiente'];condicao=rb['fps_loop']>=a.fps_alvo
texto=f"""# Relatório do projeto integrador

## 1. Dados e ambiente
Imagem: {a.imagem}. Vídeo: {a.video}.
Hardware: {hw['cpu']}; placa: {hw['modelo_placa']}; SO: {hw['sistema']}.
OpenCV {hw['opencv']}; Python {hw['python']}; threads {hw['threads_opencv']}.
Registrar a origem/licença das mídias no relatório. O pacote fornece KITTI e Udacity.

## 2. Item A
Resolução (W,H): {ra['metadados']['resolucao_largura_altura']}.
dtype: {ra['metadados']['dtype']}; canais: {ra['metadados']['canais']}.
Bytes da matriz: {ra['metadados']['bytes_pixels']}; bytes do arquivo: {ra['metadados']['bytes_arquivo']}.
Brilho dos quadrantes (cinza): {ra['brilho_quadrantes']}.
CLAHE: clipLimit=2 e tileGridSize=(8,8). Figuras em item_a/.
Discutir onde o contraste melhorou visualmente e onde o ruído foi reforçado.

## 3. Item B
Pipeline: resize 640×480 → HSV → V → bilateral d={a.d}, sigmas {a.sigma_cor}/{a.sigma_espaco}.
Frames medidos: {rb['frames_medidos']}; FPS nominal do vídeo: {rb['fps_nominal_video']:.2f}.
Média do processamento: {rb['proc_media_ms']:.3f} ms; FPS processamento: {rb['fps_processamento']:.2f}.
FPS do loop: {rb['fps_loop']:.2f}; p95/p99 do loop: {rb['loop_p95_ms']:.3f}/{rb['loop_p99_ms']:.3f} ms.
Meta: {a.fps_alvo:g} FPS; prazo: {rb['prazo_ms']:.3f} ms.
Percentual de frames acima do prazo: {rb['percentual_acima_prazo']:.2f}%.
A taxa média offline do loop {'atingiu' if condicao else 'não atingiu'} a meta nesta execução.
O loop inclui gravação/impressão; veja o escopo completo em item_b/resumo.json.

## 4. Viabilidade embarcada
Esta medição pertence ao hardware descrito acima. Se não foi feita no Raspberry
Pi 4, a viabilidade nessa placa permanece por verificar. Repetir com o mesmo
vídeo, parâmetros e protocolo; registrar clock, temperatura e throttling.
Confrontar FPS do fluxo, percentis e deadlines com a meta da aplicação, deixando
margem para detecção/rastreamento. Não extrapolar os números de um PC ou Colab.

## 5. Interpretação técnica
Sombras diminuem o contraste faixa/asfalto e obstáculo/fundo; saturação perde
detalhes e compressão modifica transições. CLAHE redistribui contraste local e
bilateral reduz variações locais preservando algumas bordas. Comparar imagens
e métricas; a melhoria da percepção só pode ser afirmada após validação própria.
"""
(out/'relatorio.md').write_text(texto,encoding='utf-8');print(texto)

# Exibir as figuras salvas na saída da célula.
for figura in sorted(out.rglob('*.png')):
    display(Image(filename=str(figura), width=850))

# MP4V nem sempre é reproduzido pelo navegador. Converta para H.264 no Colab.
import shutil
feed = out/('item_b/feed_processado.mp4' if 'projeto_integrador' in out.name else 'feed_processado.mp4')
if feed.exists() and shutil.which('ffmpeg'):
    navegador = feed.with_name('feed_navegador.mp4')
    subprocess.run(['ffmpeg','-y','-loglevel','error','-i',str(feed),'-c:v','libx264','-pix_fmt','yuv420p','-movflags','+faststart',str(navegador)],check=True)
    display(Video(str(navegador), embed=True, width=900))
else:
    print('Feed salvo:', feed, '| Abra no VLC ou instale FFmpeg para exibir no navegador.')


## Executar com seus dados
Troque `sys.argv` da célula desejada, por exemplo:
`[__file__, '--sem-gui', '--imagem', '/content/minha_foto.jpg', '--video', '/content/meu_video.mp4', '--quadros', '300', '--aquecimento', '20']`.

O runtime do Colab não corresponde ao Raspberry Pi 4. Para a discussão de viabilidade, execute o mesmo script na placa e compare sob o mesmo protocolo.
